# DOM Trade — Analysis & Visualisation Notebook
**Run this after `main.py` has completed.**

Generates all plots for documentation and presentation.

### Prerequisites
```
reliance_features.parquet          (from feature_engine.py)
regime/reliance_regimes.parquet    (from regime_detector_v2.py)
regime/regime_detector.pkl         (saved regime model)
```

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import sys, os, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

# ── Style ─────────────────────────────────────────────────────────────────────
plt.style.use('dark_background')

ACCENT  = '#00ff88'
RED     = '#ff4455'
YELLOW  = '#ffd700'
BLUE    = '#4488ff'
PURPLE  = '#cc88ff'
MUTED   = '#555570'
BG      = '#0a0a0f'
SURFACE = '#111118'
TEXT    = '#e0e0e0'

PALETTE       = [ACCENT, RED, YELLOW, BLUE, PURPLE, '#ff8844', '#44ffdd']
REGIME_COLORS = {0: ACCENT, 1: RED, 2: YELLOW}
REGIME_NAMES  = {0: 'Bullish', 1: 'Bearish', 2: 'Volatile'}

def fig_style(fig, ax_list=None):
    fig.patch.set_facecolor(BG)
    if ax_list:
        for ax in (ax_list if isinstance(ax_list, list) else [ax_list]):
            ax.set_facecolor(SURFACE)
            ax.tick_params(colors=TEXT, labelsize=9)
            ax.xaxis.label.set_color(TEXT)
            ax.yaxis.label.set_color(TEXT)
            ax.title.set_color(TEXT)
            for spine in ax.spines.values():
                spine.set_edgecolor(MUTED)

os.makedirs('plots', exist_ok=True)
print('Setup complete. Plots will save to ./plots/')


## 1. Load Data

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────
FEATURES_PATH = r'C:\project\reliance_features.parquet'
REGIMES_PATH  = r'C:\project\regime\reliance_regimes.parquet'
df = pd.read_parquet(FEATURES_PATH)

# Merge regime labels if available
if os.path.exists(REGIMES_PATH):
    dfr = pd.read_parquet(REGIMES_PATH)
    if 'regime' not in df.columns and 'regime' in dfr.columns:
        merge_cols = ['regime']
        if 'time' in dfr.columns and 'time' in df.columns:
            df = df.merge(dfr[['time','regime']], on='time', how='left')
        else:
            df['regime'] = dfr['regime'].values[:len(df)]
    elif 'regime' in dfr.columns and 'regime' not in df.columns:
        df['regime'] = dfr['regime'].values[:len(df)]
else:
    print('[warn] regime/reliance_regimes.parquet not found — regime plots will be skipped')

if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'], utc=True)
    df = df.sort_values('time').reset_index(drop=True)

FEATURE_COLS = [
    'mid_price','spread','spread_pct','ltp_mid_delta','vwmp',
    'obi_l1','obi','depth_ratio','ask_depth','bid_depth',
    'vwap_pressure','bid_concentration','ask_concentration',
    'fill_price_buy','fill_price_sell',
    'support_price','resistance_price','dist_to_support','dist_to_resistance',
    'support_strength','resistance_strength',
    'realized_vol_1t','realized_vol_5t','realized_vol_15t','realized_vol_30t','realized_vol_60t',
    'rolling_return_1t','rolling_return_5t','rolling_return_15t','rolling_return_30t','rolling_return_60t',
    'obi_mean_1t','obi_mean_5t','obi_mean_15t','obi_mean_30t','obi_mean_60t',
    'spread_vol_1t','spread_vol_5t','spread_vol_15t','spread_vol_30t','spread_vol_60t',
    'volume_spike_1t','volume_spike_5t','volume_spike_15t','volume_spike_30t','volume_spike_60t',
]

FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

print(f'Loaded : {len(df):,} rows | {len(FEATURE_COLS)} features')
print(f'Columns: {list(df.columns[:10])} ...')

regime_info = df['regime'].value_counts().to_dict() if 'regime' in df.columns else 'not available'
print(f'Regimes: {regime_info}')
print(f'Target : UP={df["target"].sum():,}  DOWN={(df["target"]==0).sum():,}')


## 2. Full Day Price Overview

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig_style(fig, axes.tolist())
fig.suptitle('RELIANCE-EQ — Full Day Overview', color=TEXT, fontsize=14, fontweight='bold')

axes[0].plot(df.index, df['mid_price'], color=BLUE, lw=1.2, label='Mid Price')
if 'vwmp' in df.columns:
    axes[0].plot(df.index, df['vwmp'], color=MUTED, lw=0.8, alpha=0.6, label='VWMP')
axes[0].set_ylabel('Price (Rs)', color=TEXT)
axes[0].legend(loc='upper right', fontsize=8)
axes[0].set_title('Mid Price vs VWMP', color=TEXT)

axes[1].plot(df.index, df['obi'], color=ACCENT, lw=0.8, alpha=0.9)
axes[1].axhline(0, color=MUTED, lw=0.8, linestyle='--')
axes[1].fill_between(df.index, df['obi'], 0,
    where=df['obi']>0, alpha=0.3, color=ACCENT, label='Bid pressure')
axes[1].fill_between(df.index, df['obi'], 0,
    where=df['obi']<0, alpha=0.3, color=RED, label='Ask pressure')
axes[1].set_ylabel('OBI', color=TEXT)
axes[1].set_ylim(-1, 1)
axes[1].legend(loc='upper right', fontsize=8)
axes[1].set_title('Order Book Imbalance (Full 5-Level)', color=TEXT)

vol_col = 'volume_spike_1t' if 'volume_spike_1t' in df.columns else None
if vol_col:
    axes[2].bar(df.index, df[vol_col].clip(lower=0), color=PURPLE, alpha=0.6, width=1)
    axes[2].set_ylabel('Volume Spike', color=TEXT)
else:
    axes[2].text(0.5, 0.5, 'Volume column not available', transform=axes[2].transAxes,
                 ha='center', color=TEXT)
axes[2].set_xlabel('Tick Index', color=TEXT)
axes[2].set_title('Volume Activity', color=TEXT)

plt.tight_layout()
plt.savefig('plots/01_full_day_overview.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: plots/01_full_day_overview.png')


## 3. Feature Distributions

In [ ]:
key_features = [
    'obi', 'obi_l1', 'spread_pct', 'vwap_pressure',
    'depth_ratio', 'bid_concentration', 'ask_concentration',
    'ltp_mid_delta', 'dist_to_support', 'dist_to_resistance',
    'realized_vol_60t', 'rolling_return_60t']
key_features = [f for f in key_features if f in df.columns]

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig_style(fig, axes.flatten().tolist())
fig.suptitle('Feature Distributions — RELIANCE-EQ', color=TEXT, fontsize=14, fontweight='bold')

for i, feat in enumerate(key_features):
    ax = axes[i//4][i%4]
    data = df[feat].dropna()
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data.clip(p1, p99)
    ax.hist(data_clipped, bins=50, color=PALETTE[i % len(PALETTE)], alpha=0.8, edgecolor='none')
    ax.axvline(data.mean(), color='white', lw=1.5, linestyle='--', label=f'mean={data.mean():.4f}')
    ax.set_title(feat, color=TEXT, fontsize=9)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('plots/02_feature_distributions.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: plots/02_feature_distributions.png')


## 4. Feature Correlation Heatmap

In [ ]:
corr_features = [
    'obi','obi_l1','spread_pct','vwap_pressure','depth_ratio',
    'bid_concentration','ask_concentration','ltp_mid_delta',
    'realized_vol_15t','realized_vol_60t',
    'rolling_return_15t','rolling_return_60t',
    'obi_mean_15t','obi_mean_60t',
    'spread_vol_15t','volume_spike_15t',
    'dist_to_support','dist_to_resistance',
    'support_strength','resistance_strength',
    'target']
corr_features = [f for f in corr_features if f in df.columns]
corr_matrix   = df[corr_features].corr()

cmap = LinearSegmentedColormap.from_list('dom_trade', [RED, BG, ACCENT], N=256)
fig, ax = plt.subplots(figsize=(16, 14))
fig_style(fig, ax)

sns.heatmap(
    corr_matrix, ax=ax, cmap=cmap, center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    linewidths=0.3, linecolor=BG,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'}, square=True)

if 'target' in corr_features:
    ti = corr_features.index('target')
    ax.axhline(ti, color=YELLOW, lw=1.5, alpha=0.7)
    ax.axvline(ti, color=YELLOW, lw=1.5, alpha=0.7)

ax.set_title('Feature Correlation Matrix', color=TEXT, fontsize=13, pad=15)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)

plt.tight_layout()
plt.savefig('plots/03_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

target_corr = corr_matrix['target'].drop('target').abs().sort_values(ascending=False)
print('Top 10 features by |correlation| with target:')
print(target_corr.head(10).to_string())


## 5. OBI Deep Dive

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig_style(fig, axes.flatten().tolist())
fig.suptitle('Order Book Imbalance — Deep Analysis', color=TEXT, fontsize=14, fontweight='bold')

sample = df.sample(min(3000, len(df)), random_state=42)
sc = axes[0][0].scatter(sample['obi_l1'], sample['obi'],
    c=sample['target'].astype(float), cmap='RdYlGn', alpha=0.4, s=8)
axes[0][0].axhline(0, color=MUTED, lw=0.8, linestyle='--')
axes[0][0].axvline(0, color=MUTED, lw=0.8, linestyle='--')
axes[0][0].set_xlabel('OBI Level-1', color=TEXT)
axes[0][0].set_ylabel('OBI Full 5-Level', color=TEXT)
axes[0][0].set_title('OBI L1 vs OBI 5-Level (color=target)', color=TEXT)
plt.colorbar(sc, ax=axes[0][0], label='Target (1=UP)')

obi_bins = pd.cut(df['obi'], bins=20)
target_by_obi = df.groupby(obi_bins, observed=True)['target'].mean()
colors_bar = [ACCENT if v > 0.5 else RED for v in target_by_obi.values]
axes[0][1].bar(range(len(target_by_obi)), target_by_obi.values, color=colors_bar, alpha=0.8)
axes[0][1].axhline(0.5, color=YELLOW, lw=1.5, linestyle='--', label='50% baseline')
axes[0][1].set_xlabel('OBI Bin (low to high)', color=TEXT)
axes[0][1].set_ylabel('P(price goes UP)', color=TEXT)
axes[0][1].set_title('OBI Bin vs Probability of UP Move', color=TEXT)
axes[0][1].legend(fontsize=8)

n = min(2000, len(df))
axes[1][0].plot(range(n), df['obi_mean_1t'].iloc[:n],  color=MUTED,  lw=0.6, alpha=0.7, label='1t')
axes[1][0].plot(range(n), df['obi_mean_15t'].iloc[:n], color=BLUE,   lw=1.2, alpha=0.9, label='15t')
axes[1][0].plot(range(n), df['obi_mean_60t'].iloc[:n], color=ACCENT, lw=2.0, label='60t')
axes[1][0].axhline(0, color=MUTED, lw=0.8, linestyle='--')
axes[1][0].set_title('Multi-Scale OBI Mean', color=TEXT)
axes[1][0].set_ylabel('OBI Mean', color=TEXT)
axes[1][0].legend(fontsize=8)

axes[1][1].hist(df[df['target']==1]['obi'].dropna(), bins=60, color=ACCENT,
    alpha=0.6, density=True, label='UP ticks')
axes[1][1].hist(df[df['target']==0]['obi'].dropna(), bins=60, color=RED,
    alpha=0.6, density=True, label='DOWN ticks')
axes[1][1].set_xlabel('OBI', color=TEXT)
axes[1][1].set_ylabel('Density', color=TEXT)
axes[1][1].set_title('OBI Distribution: UP vs DOWN', color=TEXT)
axes[1][1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('plots/04_obi_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: plots/04_obi_analysis.png')


## 6. Regime Analysis

In [ ]:
if 'regime' not in df.columns:
    print('No regime column — run regime_detector_v2.py first and re-load data.')
else:
    fig = plt.figure(figsize=(18, 12))
    fig.patch.set_facecolor(BG)
    gs  = gridspec.GridSpec(3, 3, figure=fig)
    fig.suptitle('Regime Detection Analysis', color=TEXT, fontsize=14, fontweight='bold')

    ax1 = fig.add_subplot(gs[0, :])
    ax1.set_facecolor(SURFACE)
    for r, rname in REGIME_NAMES.items():
        mask = df['regime'] == r
        if mask.sum() > 0:
            ax1.scatter(df.index[mask], df.loc[mask,'mid_price'],
                       c=REGIME_COLORS[r], s=2, alpha=0.7, label=rname)
    ax1.set_ylabel('Mid Price (Rs)', color=TEXT)
    ax1.set_title('Price Timeline Colored by Regime', color=TEXT)
    ax1.legend(markerscale=5, fontsize=9)
    ax1.tick_params(colors=TEXT)
    for sp in ax1.spines.values(): sp.set_edgecolor(MUTED)

    ax2 = fig.add_subplot(gs[1, 0])
    ax2.set_facecolor(SURFACE)
    counts = df['regime'].value_counts().sort_index()
    valid  = [(r, REGIME_NAMES[r], REGIME_COLORS[r]) for r in counts.index if r in REGIME_NAMES]
    wedges, texts, autotexts = ax2.pie(
        [counts[ri] for ri,_,_ in valid],
        labels=[f'{rn}\n{counts[ri]:,}' for ri,rn,_ in valid],
        colors=[rc for _,_,rc in valid],
        autopct='%1.1f%%', startangle=90,
        textprops={'color': TEXT, 'fontsize': 9}
    )
    for at in autotexts: at.set_color(BG)
    ax2.set_title('Regime Distribution', color=TEXT)

    ax3 = fig.add_subplot(gs[1, 1])
    ax3.set_facecolor(SURFACE)
    for r, rname in REGIME_NAMES.items():
        mask = df['regime'] == r
        if mask.sum() > 0:
            ax3.hist(df.loc[mask,'realized_vol_60t'].dropna(), bins=40,
                    color=REGIME_COLORS[r], alpha=0.6, density=True, label=rname)
    ax3.set_xlabel('Realized Vol (60t)', color=TEXT)
    ax3.set_title('Volatility by Regime', color=TEXT)
    ax3.legend(fontsize=8)
    ax3.tick_params(colors=TEXT)
    for sp in ax3.spines.values(): sp.set_edgecolor(MUTED)

    ax4 = fig.add_subplot(gs[1, 2])
    ax4.set_facecolor(SURFACE)
    for r, rname in REGIME_NAMES.items():
        mask = df['regime'] == r
        if mask.sum() > 0:
            ax4.hist(df.loc[mask,'rolling_return_60t'].dropna()*100, bins=40,
                    color=REGIME_COLORS[r], alpha=0.6, density=True, label=rname)
    ax4.set_xlabel('Rolling Return 60t (%)', color=TEXT)
    ax4.set_title('Return Distribution by Regime', color=TEXT)
    ax4.legend(fontsize=8)
    ax4.tick_params(colors=TEXT)
    for sp in ax4.spines.values(): sp.set_edgecolor(MUTED)

    ax5 = fig.add_subplot(gs[2, :])
    ax5.set_facecolor(SURFACE)
    for r, rname in REGIME_NAMES.items():
        mask = df['regime'] == r
        if mask.sum() > 0:
            ax5.scatter(df.index[mask], df.loc[mask,'obi'],
                       c=REGIME_COLORS[r], s=2, alpha=0.5, label=rname)
    ax5.axhline(0, color=MUTED, lw=0.8, linestyle='--')
    ax5.set_ylabel('OBI', color=TEXT)
    ax5.set_xlabel('Tick Index', color=TEXT)
    ax5.set_title('OBI Colored by Detected Regime', color=TEXT)
    ax5.legend(markerscale=5, fontsize=9)
    ax5.tick_params(colors=TEXT)
    for sp in ax5.spines.values(): sp.set_edgecolor(MUTED)

    plt.tight_layout()
    plt.savefig('plots/05_regime_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
    print('Saved: plots/05_regime_analysis.png')


## 7. Multi-Scale Feature Analysis

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig_style(fig, axes.flatten().tolist())
fig.suptitle('Multi-Scale Feature Analysis (5 Windows)', color=TEXT, fontsize=14, fontweight='bold')

windows  = [1, 5, 15, 30, 60]
colors_w = [MUTED, BLUE, YELLOW, RED, ACCENT]
n        = min(1500, len(df))

for w, c in zip(windows, colors_w):
    col = f'realized_vol_{w}t'
    if col in df.columns:
        axes[0][0].plot(range(n), df[col].iloc[:n], color=c, lw=1.2, alpha=0.85, label=f'{w}t')
axes[0][0].set_title('Realized Volatility — All Windows', color=TEXT)
axes[0][0].set_ylabel('Std(mid_price)', color=TEXT)
axes[0][0].legend(fontsize=8)

for w, c in zip(windows, colors_w):
    col = f'rolling_return_{w}t'
    if col in df.columns:
        axes[0][1].plot(range(n), df[col].iloc[:n]*100, color=c, lw=1.0, alpha=0.85, label=f'{w}t')
axes[0][1].axhline(0, color=MUTED, lw=0.8, linestyle='--')
axes[0][1].set_title('Rolling Return (%) — All Windows', color=TEXT)
axes[0][1].set_ylabel('Return (%)', color=TEXT)
axes[0][1].legend(fontsize=8)

for w, c in zip(windows, colors_w):
    col = f'obi_mean_{w}t'
    if col in df.columns:
        axes[1][0].plot(range(n), df[col].iloc[:n], color=c, lw=1.2, alpha=0.85, label=f'{w}t')
axes[1][0].axhline(0, color=MUTED, lw=0.8, linestyle='--')
axes[1][0].set_title('OBI Mean — All Windows', color=TEXT)
axes[1][0].set_ylabel('Mean OBI', color=TEXT)
axes[1][0].legend(fontsize=8)

for w, c in zip(windows, colors_w):
    col = f'spread_vol_{w}t'
    if col in df.columns:
        axes[1][1].plot(range(n), df[col].iloc[:n], color=c, lw=1.0, alpha=0.85, label=f'{w}t')
axes[1][1].set_title('Spread Volatility — All Windows', color=TEXT)
axes[1][1].set_ylabel('Std(spread)', color=TEXT)
axes[1][1].legend(fontsize=8)

for w, c in zip([5,15,30,60], [BLUE,YELLOW,RED,ACCENT]):
    col = f'volume_spike_{w}t'
    if col in df.columns:
        axes[2][0].plot(range(n), df[col].iloc[:n], color=c, lw=1.0, alpha=0.85, label=f'{w}t')
axes[2][0].axhline(0, color=MUTED, lw=0.8, linestyle='--')
axes[2][0].set_title('Volume Spike — Selected Windows', color=TEXT)
axes[2][0].legend(fontsize=8)

ms_features = [f'{m}_{w}t' for m in ['realized_vol','rolling_return','obi_mean','spread_vol','volume_spike']
               for w in windows if f'{m}_{w}t' in df.columns]
target_corr = df[ms_features + ['target']].corr()['target'].drop('target')
colors_bar  = [ACCENT if v > 0 else RED for v in target_corr.values]
axes[2][1].barh(range(len(target_corr)), target_corr.values, color=colors_bar, alpha=0.8)
axes[2][1].set_yticks(range(len(target_corr)))
axes[2][1].set_yticklabels(target_corr.index, fontsize=7)
axes[2][1].axvline(0, color=MUTED, lw=0.8)
axes[2][1].set_xlabel('Correlation with Target', color=TEXT)
axes[2][1].set_title('Multi-Scale Feature vs Target', color=TEXT)

plt.tight_layout()
plt.savefig('plots/06_multiscale_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: plots/06_multiscale_analysis.png')


## 8. Support & Resistance Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig_style(fig, axes.flatten().tolist())
fig.suptitle('Support & Resistance — Live Order Book Levels', color=TEXT, fontsize=14, fontweight='bold')

n = min(2000, len(df))
axes[0][0].plot(range(n), df['mid_price'].iloc[:n], color=BLUE, lw=1.5, label='Mid Price', zorder=3)
axes[0][0].plot(range(n), df['support_price'].iloc[:n], color=ACCENT, lw=0.8, alpha=0.7,
               linestyle='--', label='Live Support')
axes[0][0].plot(range(n), df['resistance_price'].iloc[:n], color=RED, lw=0.8, alpha=0.7,
               linestyle='--', label='Live Resistance')
axes[0][0].fill_between(range(n), df['support_price'].iloc[:n], df['resistance_price'].iloc[:n],
    alpha=0.07, color=YELLOW)
axes[0][0].set_title('Price with Live Support/Resistance', color=TEXT)
axes[0][0].set_ylabel('Price (Rs)', color=TEXT)
axes[0][0].legend(fontsize=8)

axes[0][1].hist(df['dist_to_support'].clip(-5,5).dropna(), bins=60, color=ACCENT, alpha=0.8)
axes[0][1].axvline(0, color=YELLOW, lw=1.5, linestyle='--')
axes[0][1].set_xlabel('Distance to Support (Rs)', color=TEXT)
axes[0][1].set_title('Distribution: Distance to Live Support', color=TEXT)

axes[1][0].hist(df['dist_to_resistance'].clip(-5,5).dropna(), bins=60, color=RED, alpha=0.8)
axes[1][0].axvline(0, color=YELLOW, lw=1.5, linestyle='--')
axes[1][0].set_xlabel('Distance to Resistance (Rs)', color=TEXT)
axes[1][0].set_title('Distribution: Distance to Live Resistance', color=TEXT)

axes[1][1].plot(range(n), df['support_strength'].iloc[:n], color=ACCENT, lw=1.2, label='Support Strength')
axes[1][1].plot(range(n), df['resistance_strength'].iloc[:n], color=RED, lw=1.2, label='Resistance Strength')
axes[1][1].set_title('Rolling Wall Strength (50-tick mean qty)', color=TEXT)
axes[1][1].set_ylabel('Mean Qty at Level', color=TEXT)
axes[1][1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('plots/07_support_resistance.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: plots/07_support_resistance.png')


## 9. Run Backtest & Collect Results

In [ ]:
from regime.regime_detector_v2 import RegimeDetector
from regime.forecasters import Forecasters
from regime.ensemble import Ensemble

REGIME_MODEL = 'regime/regime_detector.pkl'
PARQUET      = 'regime/reliance_regimes.parquet'

df_bt = pd.read_parquet(PARQUET)
df_bt = df_bt.dropna(subset=['target']).reset_index(drop=True)

cut   = int(len(df_bt) * 0.8)
train = df_bt.iloc[:cut].reset_index(drop=True)
test  = df_bt.iloc[cut:].reset_index(drop=True)

detector = RegimeDetector.load(REGIME_MODEL)
fc       = Forecasters()
ens      = Ensemble(fc, detector)

print(f'Warming up on {len(train):,} training ticks...')
for _, row in train.iterrows():
    ens.fc.learn_all(row.to_dict(), int(row['target']))

print(f'Replaying {len(test):,} test ticks...')
results = []
for _, row in test.iterrows():
    result = ens.step(row.to_dict(), int(row['target']))
    result['true_label'] = int(row['target'])
    result['mid_price']  = row.get('mid_price', 0)
    result['obi']        = row.get('obi', 0)
    results.append(result)

res     = pd.DataFrame(results)
y_true  = res['true_label'].values
y_pred  = res['final_pred'].values
y_ftrl  = res['pred_ftrl'].values
y_hoeff = res['pred_hoeff'].values
y_pa    = res['pred_pa'].values
regimes = res['regime'].values
lats    = res['latency_ns'].values.astype(float)

print(f'Ensemble accuracy : {(y_pred==y_true).mean():.4f}')
print(f'Drift events      : {ens.drift_count}')


## 10. Model Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig_style(fig, axes.flatten().tolist())
fig.suptitle('Model Comparison — Backtest Results', color=TEXT, fontsize=14, fontweight='bold')

model_names  = ['Naive\n(always UP)', 'FTRL', 'Hoeffding\nTree', 'PA', 'Ensemble']
model_colors = [MUTED, BLUE, YELLOW, RED, ACCENT]

model_accs   = [
    (np.ones_like(y_true) == y_true).mean(),
    (y_ftrl  == y_true).mean(),
    (y_hoeff == y_true).mean(),
    (y_pa    == y_true).mean(),
    (y_pred  == y_true).mean(),
]

bars = axes[0][0].bar(model_names, model_accs, color=model_colors, alpha=0.85, width=0.6)
for bar, acc in zip(bars, model_accs):
    axes[0][0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
        f'{acc:.2%}', ha='center', va='bottom', color=TEXT, fontsize=9, fontweight='bold')
axes[0][0].axhline(model_accs[0], color=MUTED, lw=1.5, linestyle='--', alpha=0.7, label='Naive')
axes[0][0].set_ylim(0, 1.12)
axes[0][0].set_ylabel('Accuracy', color=TEXT)
axes[0][0].set_title('Accuracy Comparison', color=TEXT)
axes[0][0].legend(fontsize=8)

window = 100
def roll_acc(preds, truth, w=window):
    return [(preds[max(0,i-w):i+1]==truth[max(0,i-w):i+1]).mean() for i in range(len(preds))]

axes[0][1].plot(roll_acc(y_ftrl,  y_true), color=BLUE,   lw=1.5, alpha=0.9, label='FTRL')
axes[0][1].plot(roll_acc(y_hoeff, y_true), color=YELLOW, lw=1.5, alpha=0.9, label='Hoeffding')
axes[0][1].plot(roll_acc(y_pa,    y_true), color=RED,    lw=1.5, alpha=0.9, label='PA')
axes[0][1].plot(roll_acc(y_pred,  y_true), color=ACCENT, lw=2.5, label='Ensemble')
axes[0][1].axhline(model_accs[0], color=MUTED, lw=1, linestyle='--', label='Naive')
axes[0][1].set_xlabel('Test Tick', color=TEXT)
axes[0][1].set_ylabel('Rolling Accuracy (100t window)', color=TEXT)
axes[0][1].set_title('Rolling Accuracy Over Test Set', color=TEXT)
axes[0][1].legend(fontsize=8)
axes[0][1].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.0%}'))

reg_accs, reg_labels, reg_colors = [], [], []
for r, rname in REGIME_NAMES.items():
    mask = regimes == r
    if mask.sum() > 0:
        reg_accs.append((y_pred[mask]==y_true[mask]).mean())
        reg_labels.append(f'{rname}\n({mask.sum()} ticks)')
        reg_colors.append(REGIME_COLORS[r])

bars2 = axes[1][0].bar(reg_labels, reg_accs, color=reg_colors, alpha=0.85, width=0.5)
for bar, acc in zip(bars2, reg_accs):
    axes[1][0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
        f'{acc:.2%}', ha='center', va='bottom', color=TEXT, fontsize=10, fontweight='bold')
axes[1][0].axhline(model_accs[0], color=MUTED, lw=1.5, linestyle='--', label='Naive baseline')
axes[1][0].set_ylim(0, 1.12)
axes[1][0].set_title('Ensemble Accuracy per Regime', color=TEXT)
axes[1][0].legend(fontsize=8)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
cmap_cm = LinearSegmentedColormap.from_list('cm', [BG, ACCENT], N=256)
im = axes[1][1].imshow(cm, cmap=cmap_cm, aspect='auto')
axes[1][1].set_xticks([0,1]); axes[1][1].set_yticks([0,1])
axes[1][1].set_xticklabels(['Pred DOWN','Pred UP'], color=TEXT)
axes[1][1].set_yticklabels(['True DOWN','True UP'], color=TEXT)
for i in range(2):
    for j in range(2):
        axes[1][1].text(j, i, f'{cm[i,j]:,}\n({cm[i,j]/cm.sum():.1%})',
            ha='center', va='center', color=TEXT, fontsize=11, fontweight='bold')
axes[1][1].set_title('Confusion Matrix (Ensemble)', color=TEXT)
plt.colorbar(im, ax=axes[1][1])

plt.tight_layout()
plt.savefig('plots/08_model_comparison.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: plots/08_model_comparison.png')


## 11. Latency Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig_style(fig, axes.tolist())
fig.suptitle('Latency Analysis — Full Pipeline', color=TEXT, fontsize=14, fontweight='bold')

lats_us = lats / 1000
axes[0].hist(lats_us, bins=80, color=ACCENT, alpha=0.85, edgecolor='none')
axes[0].axvline(np.mean(lats_us), color=YELLOW, lw=2, linestyle='--',
    label=f'Mean: {np.mean(lats_us):.0f}us')
axes[0].axvline(np.percentile(lats_us,99), color=RED, lw=2, linestyle='--',
    label=f'p99: {np.percentile(lats_us,99):.0f}us')
axes[0].set_xlabel('Latency (us)', color=TEXT)
axes[0].set_ylabel('Count', color=TEXT)
axes[0].set_title('Latency Distribution', color=TEXT)
axes[0].legend(fontsize=9)

axes[1].plot(lats_us, color=BLUE, lw=0.8, alpha=0.8)
axes[1].axhline(np.mean(lats_us), color=YELLOW, lw=1.5, linestyle='--',
    label=f'Mean: {np.mean(lats_us):.0f}us')
axes[1].set_xlabel('Prediction Index', color=TEXT)
axes[1].set_ylabel('Latency (us)', color=TEXT)
axes[1].set_title('Latency Over Time', color=TEXT)
axes[1].legend(fontsize=9)

percentiles = [50, 75, 90, 95, 99, 99.9]
values      = [np.percentile(lats_us, p) for p in percentiles]
bar_colors  = [ACCENT if v < 500 else YELLOW if v < 1000 else RED for v in values]
bars3 = axes[2].bar([f'p{p}' for p in percentiles], values, color=bar_colors, alpha=0.85)
for bar, val in zip(bars3, values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
        f'{val:.0f}us', ha='center', va='bottom', color=TEXT, fontsize=9)
axes[2].set_ylabel('Latency (us)', color=TEXT)
axes[2].set_title('Latency Percentiles', color=TEXT)

plt.tight_layout()
plt.savefig('plots/09_latency_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Mean: {np.mean(lats_us):.0f}us  |  p99: {np.percentile(lats_us,99):.0f}us  |  Max: {np.max(lats_us):.0f}us')


## 12. Feature Importance (LightGBM offline reference)

In [ ]:
try:
    from lightgbm import LGBMClassifier
    X   = df[FEATURE_COLS].fillna(0)
    y_l = df['target'].astype(int)
    cut_lgb = int(len(X)*0.8)
    
    lgb = LGBMClassifier(n_estimators=200, learning_rate=0.05,
                         num_leaves=31, random_state=42, verbose=-1)
    lgb.fit(X.iloc[:cut_lgb], y_l.iloc[:cut_lgb])
    
    imp    = pd.Series(lgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
    top30  = imp.tail(30)
    
    c_map  = {'obi': ACCENT, 'vol': YELLOW, 'return': BLUE, 'mid': BLUE,
               'support': PURPLE, 'resist': PURPLE, 'spread': RED}
    
    def feat_color(f):
        for k,v in c_map.items():
            if k in f: return v
        return MUTED
        
    fig, ax = plt.subplots(figsize=(12, 10))
    fig_style(fig, ax)
    
    ax.barh(range(len(top30)), top30.values,
            color=[feat_color(f) for f in top30.index], alpha=0.85)
    ax.set_yticks(range(len(top30)))
    ax.set_yticklabels(top30.index, fontsize=9)
    ax.set_xlabel('Feature Importance (LightGBM gain)', color=TEXT)
    ax.set_title('Top 30 Feature Importance', color=TEXT)
    
    legend_items = [
        mpatches.Patch(color=ACCENT,  label='OBI features'),
        mpatches.Patch(color=YELLOW,  label='Volatility features'),
        mpatches.Patch(color=BLUE,    label='Price/Return features'),
        mpatches.Patch(color=PURPLE,  label='Support/Resistance'),
        mpatches.Patch(color=RED,     label='Spread features'),
        mpatches.Patch(color=MUTED,   label='Other'),
    ]
    ax.legend(handles=legend_items, loc='lower right', fontsize=8)
    
    plt.tight_layout()
    plt.savefig('plots/10_feature_importance.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
    print('Saved: plots/10_feature_importance.png')
    
except ImportError:
    print('LightGBM not installed. Run: pip install lightgbm')


## 13. Final Summary

In [ ]:
print('=' * 55)
print('DOM TRADE — COMPLETE RESULTS SUMMARY')
print('=' * 55)
print(f'Dataset  : RELIANCE-EQ (NSE) | {len(df):,} rows | {len(FEATURE_COLS)} features')
print(f'Split    : Train={cut:,} | Test={len(test):,}')
print()
print('Accuracy')
print(f'  Naive baseline  : {(np.ones_like(y_true)==y_true).mean():.4f}')
print(f'  FTRL alone      : {(y_ftrl==y_true).mean():.4f}')
print(f'  Hoeffding alone : {(y_hoeff==y_true).mean():.4f}')
print(f'  PA alone        : {(y_pa==y_true).mean():.4f}')
print(f'  Ensemble        : {(y_pred==y_true).mean():.4f}')
print(f'  Improvement     : {(y_pred==y_true).mean()-(np.ones_like(y_true)==y_true).mean():+.4f}')
print()
print('Per-Regime Accuracy')
for r, rname in REGIME_NAMES.items():
    mask = regimes == r
    if mask.sum() > 0:
        print(f'  {rname:10s}: {(y_pred[mask]==y_true[mask]).mean():.4f} ({mask.sum():,} ticks)')
print()
print('Latency')
print(f'  Mean : {np.mean(lats/1000):.0f} us  |  p99: {np.percentile(lats/1000,99):.0f} us')
print(f'Drift Events : {ens.drift_count}')
print('=' * 55)
